In [1]:
import pandas as pd

players_df = pd.read_csv("/Users/michaelliu/.cache/kagglehub/datasets/eoinamoore/historical-nba-data-and-player-box-scores/versions/515/Players.csv")
stats_df = pd.read_csv("/Users/michaelliu/.cache/kagglehub/datasets/eoinamoore/historical-nba-data-and-player-box-scores/versions/515/PlayerStatistics.csv")
stats_df['date'] = pd.to_datetime(stats_df['gameDate'])

/var/folders/k3/vwf46zk577vd00j0_2_9hdrr0000gn/T/ipykernel_11874/1658700062.py:4: DtypeWarning: Columns (0: gameLabel, 1: gameSubLabel, 2: seriesGameNumber, 3: numMinutes, 4: comment, 5: startingPosition) have mixed types. Specify dtype option on import or set low_memory=False.
  stats_df = pd.read_csv("/Users/michaelliu/.cache/kagglehub/datasets/eoinamoore/historical-nba-data-and-player-box-scores/versions/515/PlayerStatistics.csv")


In [45]:
reg_season = stats_df[(stats_df['date'].between('2010-10-24', '2020-08-14')) & (stats_df['gameType'].isin(['Regular Season']))]
reg_season['numMinutes_num'] = pd.to_numeric(reg_season['numMinutes'], errors='coerce')
reg_season_played = reg_season[reg_season['numMinutes_num'] > 0]

In [46]:
stat_cols = [
    'points', 'assists', 'reboundsTotal', 'blocks', 'steals', 'turnovers',
    'fieldGoalsAttempted', 'fieldGoalsMade', 'threePointersAttempted', 'threePointersMade',
    'freeThrowsAttempted', 'freeThrowsMade'
]

player_averages = reg_season_played.groupby('personId').agg({
    'firstName': 'first',
    'lastName': 'first',
    'gameId': 'count',
    **{col: 'mean' for col in stat_cols}
}).reset_index().rename(columns={'gameId': 'gamesPlayed'})

player_averages['personId'] = player_averages['personId'].astype(int)

# Merge player positions (guard, forward, center) from players_df
players_positions = players_df[['personId', 'guard', 'forward', 'center']]
player_averages = player_averages.merge(players_positions, on='personId', how='left')
player_averages[['guard', 'forward', 'center']] = player_averages[['guard', 'forward', 'center']].fillna(0).astype(int)

In [47]:
# Calculate True Shooting percentage (TS%)
player_averages['trueShootingAttempts'] = player_averages['fieldGoalsAttempted'] + 0.44 * player_averages['freeThrowsAttempted']
player_averages['ts'] = player_averages['points'] / (2 * player_averages['trueShootingAttempts'])
player_averages['ts'] = player_averages['ts'].fillna(0.0)

# Calculate Fantasy Points
player_averages['fantasyPoints'] = (
    player_averages['points'] +
    1.5 * player_averages['reboundsTotal'] +
    1.5 * player_averages['assists'] +
    4 * player_averages['steals'] +
    4 * player_averages['blocks'] -
    2 * player_averages['turnovers']
)

In [48]:
# Filter out players with no valid position
player_averages = player_averages[player_averages[['guard', 'forward', 'center']].any(axis=1)]

# Filter to players who played 20 or more games
player_averages = player_averages[player_averages['gamesPlayed'] >= 20]

# Sort players by fantasyPoints descending
player_averages = player_averages.sort_values(by='fantasyPoints', ascending=False)

In [51]:
output_csv_path = 'player_averages_2010_20.csv'
player_averages.to_csv(output_csv_path, index=False)
print(f"Successfully calculated averages for {len(player_averages)} players and saved to {output_csv_path}")

Successfully calculated averages for 1031 players and saved to player_averages_2010_20.csv


In [50]:
player_averages

,personId,firstName,lastName,gamesPlayed,points,assists,reboundsTotal,blocks,steals,turnovers,...,threePointersAttempted,threePointersMade,freeThrowsAttempted,freeThrowsMade,guard,forward,center,trueShootingAttempts,ts,fantasyPoints
585,203076,Anthony,Davis,528,24.009470,2.238636,10.382576,2.393939,1.382576,1.937500,...,1.503788,0.479167,7.119318,5.712121,0,1,1,20.416591,0.587989,54.172348
127,2544,LeBron,James,717,26.485356,7.721060,7.732218,0.662483,1.478382,3.656904,...,4.336123,1.539749,7.203626,5.235704,0,1,0,21.827894,0.606686,50.914923
335,201566,Russell,Westbrook,714,24.977591,8.698880,7.578431,0.308123,1.844538,4.256303,...,4.166667,1.287115,7.529412,6.018207,1,0,0,23.311541,0.535734,49.491597
799,203954,Joel,Embiid,208,23.995192,3.125000,11.495192,1.793269,0.754808,3.514423,...,3.591346,1.144231,8.615385,6.831731,0,1,1,20.487885,0.585595,49.088942
285,201142,Kevin,Durant,613,27.688418,4.662316,7.404568,1.189233,1.079935,3.140294,...,5.442088,2.097879,7.657423,6.758564,0,1,0,22.023426,0.628613,48.584829
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
636,203129,Tornike,Shengelia,37,1.594595,0.459459,1.027027,0.054054,0.135135,0.621622,...,0.216216,0.027027,0.648649,0.324324,0,1,0,1.663784,0.479207,3.337838
492,202380,Hamady,Ndiaye,28,0.714286,0.107143,0.857143,0.321429,0.071429,0.214286,...,0.000000,0.000000,0.392857,0.214286,0,0,1,0.708571,0.504032,3.303571
1036,1628399,Tyler,Lydon,21,1.095238,0.285714,0.857143,0.000000,0.095238,0.190476,...,0.476190,0.190476,0.142857,0.047619,0,1,0,0.920000,0.595238,2.809524
101,2260,Jarron,Collins,26,0.730769,0.076923,0.923077,0.038462,0.153846,0.269231,...,0.000000,0.000000,0.384615,0.269231,0,0,1,0.976923,0.374016,2.461538
